## 1. Imports ##

In [54]:
import requests
import pandas as pd
import numpy as np
import time
from datetime import datetime
from collections import Counter

## 2. API Configuration ##

In [55]:
HEADERS = {
    "User-Agent": "WBS-ESG-Analyzer/1.0 (student project)",
    "Accept": "application/json",
    "Accept-Language": "en-US,en;q=0.9",
}

In [56]:
SOURCES = {
    "food": "https://world.openfoodfacts.org",
    "beauty": "https://world.openbeautyfacts.org",
    "products": "https://world.openproductsfacts.org"
}

In [57]:
USEFUL_FIELDS = [
    "code",
    "product_name",
    "generic_name",
    "brands",
    "categories",
    "categories_tags",
    "labels",
    "labels_tags",
    "ingredients_text",
    "ingredients_tags",
    "packaging",
    "packaging_tags",
    "countries",
    "countries_tags",
    "stores",
    "origins",
    "origins_tags",
    "manufacturing_places",
    "ecoscore_grade",
    "ecoscore_score",
    "image_url",
    "last_modified_t",
    "created_t"
]

FIELDS_PARAM = ",".join(USEFUL_FIELDS)

## 3. Unified Schema ##

In [58]:
UNIFIED_COLUMNS = [
    "source",
    "retrieval_method",
    "search_query",
    "barcode",
    "product_name",
    "generic_name",
    "brand",
    "categories",
    "category_tags",
    "labels",
    "label_tags",
    "ingredients_text",
    "ingredient_tags",
    "packaging",
    "packaging_tags",
    "countries",
    "country_tags",
    "stores",
    "origins",
    "origin_tags",
    "manufacturing_places",
    "ecoscore_grade",
    "ecoscore_score",
    "image_url",
    "created_datetime",
    "last_modified_datetime",
    "api_status",
    "extraction_timestamp"
]

## 4. Helper Functions ##

In [59]:
# Normalize Tags

def normalize_tags(value):
    if isinstance(value, list):
        return value
    if isinstance(value, str) and value.strip():
        return [tag.strip() for tag in value.split(",")]
    return []

In [60]:
# Normalize Text

def clean_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip()

In [61]:
# Convert Unix Timestamps

def unix_to_datetime(value):
    try:
        if pd.isna(value) or value == "":
            return None
        return datetime.fromtimestamp(int(value))
    except Exception:
        return None

In [62]:
# API Request with Retries

def request_json(url, params=None, retries=5, timeout=60):
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, params=params, headers=HEADERS, timeout=timeout)

            if response.status_code == 200:
                return response.json()

            if response.status_code in (429, 503) and attempt < retries:
                wait_seconds = 3 * attempt
                print(f"Temporary status {response.status_code}; retrying in {wait_seconds}s...")
                time.sleep(wait_seconds)
                continue

            return {
                "api_status": "error",
                "status_code": response.status_code,
                "url": response.url
            }

        except requests.RequestException as exc:
            if attempt < retries:
                wait_seconds = 3 * attempt
                print(f"Request error: {exc}; retrying in {wait_seconds}s...")
                time.sleep(wait_seconds)
                continue

            return {
                "api_status": "error",
                "status_code": None,
                "url": url,
                "error_message": str(exc)
            }

## 5. Germany Filter ##

In [63]:
GERMANY_COUNTRY_TAG = "en:germany"

In [64]:
# For raw API Product Dictionaries

def is_german_product_raw(product):
    country_tags = normalize_tags(product.get("countries_tags"))
    return GERMANY_COUNTRY_TAG in country_tags

In [65]:
# For unified rows

def is_german_product_unified(row):
    country_tags = row.get("country_tags", [])
    return GERMANY_COUNTRY_TAG in country_tags

## 6. Barcode Extraction ##

In [66]:
# Generic function

def extract_from_source(source_name, barcode):
    base_url = SOURCES[source_name]
    url = f"{base_url}/api/v3/product/{barcode}"

    params = {
        "fields": FIELDS_PARAM
    }

    data = request_json(url, params=params)

    if data.get("api_status") == "error":
        return {
            "source": source_name,
            "retrieval_method": "barcode",
            "search_query": "",
            "barcode": barcode,
            "api_status": "error",
            "raw_product": {}
        }

    product = data.get("product", {})

    if not product:
        return {
            "source": source_name,
            "retrieval_method": "barcode",
            "search_query": "",
            "barcode": barcode,
            "api_status": "not_found",
            "raw_product": {}
        }

    return {
        "source": source_name,
        "retrieval_method": "barcode",
        "search_query": "",
        "barcode": barcode,
        "api_status": "found",
        "raw_product": product
    }

Required Functions:

In [67]:
def extract_food(barcode):
    return extract_from_source("food", barcode)

In [68]:
def extract_beauty(barcode):
    return extract_from_source("beauty", barcode)

In [69]:
def extract_product(barcode):
    return extract_from_source("products", barcode)

## 7. Product-Name Search ##

In [70]:
# Generic search function

def search_from_source(source_name, query, page_size=100, page=1):
    base_url = SOURCES[source_name]
    url = f"{base_url}/cgi/search.pl"

    params = {
        "search_terms": query,
        "search_simple": 1,
        "action": "process",
        "json": 1,
        "page_size": page_size,
        "page": page,
        "fields": FIELDS_PARAM,
    }

    data = request_json(url, params=params)

    if data.get("api_status") == "error":
        return []

    return data.get("products", [])

Source-specific search functions:

In [71]:
def search_food(query, page_size=100, page=1):
    return search_from_source("food", query, page_size, page)

In [72]:
def search_beauty(query, page_size=100, page=1):
    return search_from_source("beauty", query, page_size, page)

In [73]:
def search_product(query, page_size=100, page=1):
    return search_from_source("products", query, page_size, page)

In [74]:
# Convert search results into extaction-like records

def make_search_record(product, source_name, query):
    return {
        "source": source_name,
        "retrieval_method": "search",
        "search_query": query,
        "barcode": product.get("code"),
        "api_status": "found",
        "raw_product": product
    }

## 8. Transform to Unified Schema ##

In [75]:
def transform_to_unified(record):
    product = record.get("raw_product", {})

    unified = {
        "source": record.get("source"),
        "retrieval_method": record.get("retrieval_method"),
        "search_query": record.get("search_query"),
        "barcode": record.get("barcode"),
        "product_name": clean_text(product.get("product_name")),
        "generic_name": clean_text(product.get("generic_name")),
        "brand": clean_text(product.get("brands")),
        "categories": clean_text(product.get("categories")),
        "category_tags": normalize_tags(product.get("categories_tags")),
        "labels": clean_text(product.get("labels")),
        "label_tags": normalize_tags(product.get("labels_tags")),
        "ingredients_text": clean_text(product.get("ingredients_text")),
        "ingredient_tags": normalize_tags(product.get("ingredients_tags")),
        "packaging": clean_text(product.get("packaging")),
        "packaging_tags": normalize_tags(product.get("packaging_tags")),
        "countries": clean_text(product.get("countries")),
        "country_tags": normalize_tags(product.get("countries_tags")),
        "stores": clean_text(product.get("stores")),
        "origins": clean_text(product.get("origins")),
        "origin_tags": normalize_tags(product.get("origins_tags")),
        "manufacturing_places": clean_text(product.get("manufacturing_places")),
        "ecoscore_grade": clean_text(product.get("ecoscore_grade")).lower(),
        "ecoscore_score": product.get("ecoscore_score"),
        "image_url": clean_text(product.get("image_url")),
        "created_datetime": unix_to_datetime(product.get("created_t")),
        "last_modified_datetime": unix_to_datetime(product.get("last_modified_t")),
        "api_status": record.get("api_status"),
        "extraction_timestamp": datetime.now()
    }

    return unified

## 9. Test Barcode Extraction ##

In [76]:
barcode_tests = [
    {"source": "food", "barcode": "4014400400007"},
    {"source": "beauty", "barcode": "3474637195809"},
    {"source": "products", "barcode": "8700216676793"}
]

In [77]:
barcode_records = []

for item in barcode_tests:
    source = item["source"]
    barcode = item["barcode"]

    if source == "food":
        extracted = extract_food(barcode)
    elif source == "beauty":
        extracted = extract_beauty(barcode)
    elif source == "products":
        extracted = extract_product(barcode)

    unified = transform_to_unified(extracted)
    barcode_records.append(unified)

barcode_test_df = pd.DataFrame(barcode_records)
barcode_test_df

,source,retrieval_method,search_query,barcode,product_name,generic_name,brand,categories,category_tags,labels,...,origins,origin_tags,manufacturing_places,ecoscore_grade,ecoscore_score,image_url,created_datetime,last_modified_datetime,api_status,extraction_timestamp
0,food,barcode,,4014400400007,Toffifee 15er,Haselnuss (10 %) in Caramel (41 %) mit Nougatc...,Storck,Confectioneries,"[en:snacks, en:sweet-snacks, en:confectioneries]",,...,"Germany, de:Njemačka","[en:germany, de:Njemačka]","berlin, deutschland, nemčija",unknown,None,https://images.openfoodfacts.org/images/produc...,2013-05-10 22:15:21,2026-06-10 00:34:28,found,2026-06-10 12:13:58.615561
1,beauty,barcode,,3474637195809,Bain Décalcifiant Réparateur,,Kérastase,Shampoos,"[en:hair, en:shampoos]",,...,,[],,unknown,None,https://images.openbeautyfacts.org/images/prod...,2025-03-09 01:22:23,2026-06-08 19:12:38,found,2026-06-10 12:13:58.792205
2,products,barcode,,8700216676793,Staubmagnet,,Swiffer,,[],,...,,[],,unknown,None,,2025-05-23 13:42:10,2026-06-09 11:49:25,found,2026-06-10 12:13:58.980668


In [78]:
# Check schema

set(UNIFIED_COLUMNS) - set(barcode_test_df.columns)

set()

Expected result: set()

Confirmed.

## 10. Build Search-Based Dataset ##

In [79]:
# Using the Day 1 query strategy

queries = {
    "food": [
        "schokolade",
        "muesli",
        "milch",
        "kaffee"
    ],
    "beauty": [
        "shampoo",
        "seife",
        "cream",
        "deo"
    ],
    "products": [
        "handy",
        "batterie",
        "waschmittel",
        "toothbrush"
    ]
}

In [80]:
# Run searches, filter for Germany, and transform

search_records = []

for source, query_list in queries.items():
    for query in query_list:
        if source == "food":
            products = search_food(query, page_size=100)
        elif source == "beauty":
            products = search_beauty(query, page_size=100)
        elif source == "products":
            products = search_product(query, page_size=100)

        german_products = [p for p in products if is_german_product_raw(p)]

        print(source, query, len(german_products))

        for product in german_products:
            record = make_search_record(product, source, query)
            unified = transform_to_unified(record)
            search_records.append(unified)

        time.sleep(1)

food schokolade 93
food muesli 10
Temporary status 503; retrying in 3s...
Temporary status 503; retrying in 6s...
Temporary status 503; retrying in 9s...
food milch 93
Temporary status 503; retrying in 3s...
Temporary status 503; retrying in 6s...
Temporary status 503; retrying in 9s...
food kaffee 96
beauty shampoo 12
beauty seife 93
beauty cream 6
beauty deo 38
products handy 7
products batterie 27
products waschmittel 49
products toothbrush 15


In [81]:
# Create combined search dataframe

search_df = pd.DataFrame(search_records)

search_df.shape

(539, 28)

## 11. Combine Barcode + Search Results ##

In [82]:
combined_df = pd.concat(
    [barcode_test_df, search_df],
    ignore_index=True
)

C:\Users\nicom\AppData\Local\Temp\ipykernel_30716\1252519603.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat(


In [83]:
# Filter final dataset to Germany-only

combined_df = combined_df[
    combined_df.apply(is_german_product_unified, axis=1)
].copy()

In [84]:
# Remove duplicates

combined_df = combined_df.drop_duplicates(
    subset=["source", "barcode"]
).copy()

In [85]:
# Check size

combined_df.shape

(541, 28)

In [86]:
combined_df["source"].value_counts()

source
food        292
beauty      150
products     99
Name: count, dtype: int64

In [87]:
combined_df

,source,retrieval_method,search_query,barcode,product_name,generic_name,brand,categories,category_tags,labels,...,origins,origin_tags,manufacturing_places,ecoscore_grade,ecoscore_score,image_url,created_datetime,last_modified_datetime,api_status,extraction_timestamp
0,food,barcode,,4014400400007,Toffifee 15er,Haselnuss (10 %) in Caramel (41 %) mit Nougatc...,Storck,Confectioneries,"[en:snacks, en:sweet-snacks, en:confectioneries]",,...,"Germany, de:Njemačka","[en:germany, de:Njemačka]","berlin, deutschland, nemčija",unknown,NaN,https://images.openfoodfacts.org/images/produc...,2013-05-10 22:15:21,2026-06-10 00:34:28,found,2026-06-10 12:13:58.615561
1,beauty,barcode,,3474637195809,Bain Décalcifiant Réparateur,,Kérastase,Shampoos,"[en:hair, en:shampoos]",,...,,[],,unknown,NaN,https://images.openbeautyfacts.org/images/prod...,2025-03-09 01:22:23,2026-06-08 19:12:38,found,2026-06-10 12:13:58.792205
2,products,barcode,,8700216676793,Staubmagnet,,Swiffer,,[],,...,,[],,unknown,NaN,,2025-05-23 13:42:10,2026-06-09 11:49:25,found,2026-06-10 12:13:58.980668
4,food,search,schokolade,20815356,Dunkle Schokolade mit ganzen Haselnüssen,,fin CARRE,fr:Barres chocolatées à la noix de coco,"[en:snacks, en:sweet-snacks, en:cocoa-and-its-...","Sustainable farming, Fairtrade cocoa, Made in ...",...,,[],,a,75.0,https://images.openfoodfacts.org/images/produc...,2018-04-12 11:10:19,2026-04-30 22:34:49,found,2026-06-10 12:14:00.799063
5,food,search,schokolade,20029838,Hazelnut Milk Chocolate,,fin CARRE,Milk chocolates with hazelnuts,"[en:snacks, en:sweet-snacks, en:cocoa-and-its-...","Fairtrade cocoa, Green Dot, Max Havelaar, UTZ ...",...,Unspecified,[en:unspecified],,c,58.0,https://images.openfoodfacts.org/images/produc...,2014-04-30 14:59:58,2026-04-24 16:08:19,found,2026-06-10 12:14:00.799077
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
537,products,search,toothbrush,4902430897204,Cross Action Indicator Toothbrush,,Oral-B,Toothbrushes,"[en:health-beauty, en:personal-care, en:oral-c...",,...,,[],,,NaN,,2026-06-04 18:24:03,2026-06-07 17:21:43,found,2026-06-10 12:14:58.081917
538,products,search,toothbrush,8720689050166,Sonicare Electric Toothbrush Series 1100 White,,Philips,Toothbrushes,"[en:health-beauty, en:personal-care, en:oral-c...",,...,,[],,,NaN,,2026-06-04 21:09:22,2026-06-07 17:19:32,found,2026-06-10 12:14:58.081933
539,products,search,toothbrush,8720689050197,Philips Sonicare Electric Toothbrush Series 11...,,Philips,Toothbrushes,"[en:health-beauty, en:personal-care, en:oral-c...",,...,,[],,,NaN,,2026-06-04 20:57:43,2026-06-07 17:12:28,found,2026-06-10 12:14:58.081948
540,products,search,toothbrush,8714789183824,Colgate 360 Deep Clean Soft Toothbrush 1 Pack,,Colgate,,[],,...,,[],,,NaN,,2026-06-04 17:44:01,2026-06-04 17:44:01,found,2026-06-10 12:14:58.081962


## 12. Standardize Eco-Score ##

In [88]:
combined_df["ecoscore_grade_clean"] = (
    combined_df["ecoscore_grade"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

In [89]:
UNKNOWN_ECOSCORE_VALUES = ["", "unknown", "not-applicable", "nan"]

combined_df["has_ecoscore"] = ~combined_df["ecoscore_grade_clean"].isin(
    UNKNOWN_ECOSCORE_VALUES
)

In [90]:
combined_df["ecoscore_score"] = pd.to_numeric(
    combined_df["ecoscore_score"],
    errors="coerce"
)

## 13. Add Data Quality Flags ##

In [91]:
def has_list_data(value):
    return isinstance(value, list) and len(value) > 0

In [92]:
combined_df["has_product_name"] = combined_df["product_name"].str.len() > 0
combined_df["has_brand"] = combined_df["brand"].str.len() > 0
combined_df["has_categories"] = combined_df["category_tags"].apply(has_list_data)
combined_df["has_labels"] = combined_df["label_tags"].apply(has_list_data)
combined_df["has_ingredients"] = (
    combined_df["ingredients_text"].str.len() > 0
) | (
    combined_df["ingredient_tags"].apply(has_list_data)
)
combined_df["has_packaging"] = (
    combined_df["packaging"].str.len() > 0
) | (
    combined_df["packaging_tags"].apply(has_list_data)
)
combined_df["has_country"] = combined_df["country_tags"].apply(has_list_data)
combined_df["has_image"] = combined_df["image_url"].str.len() > 0

## 14. Basic Data Quality Report ##

In [93]:
# API Status

combined_df["api_status"].value_counts(dropna=False)

api_status
found    541
Name: count, dtype: int64

In [94]:
# Source counts

combined_df["source"].value_counts()

source
food        292
beauty      150
products     99
Name: count, dtype: int64

In [95]:
# Completeness by source

quality_columns = [
    "has_product_name",
    "has_brand",
    "has_categories",
    "has_labels",
    "has_ingredients",
    "has_packaging",
    "has_country",
    "has_image",
    "has_ecoscore"
]

quality_report = (
    combined_df
    .groupby("source")[quality_columns]
    .mean()
    .round(3)
)

quality_report

,has_product_name,has_brand,has_categories,has_labels,has_ingredients,has_packaging,has_country,has_image,has_ecoscore
source,,,,,,,,,
beauty,0.987,0.787,0.853,0.213,0.287,0.167,1.0,0.667,0.013
food,0.986,0.990,1.000,0.863,0.952,0.675,1.0,0.997,0.716
products,1.000,0.929,0.606,0.081,0.051,0.212,1.0,0.596,0.000


In [96]:
# Missing values

empty_report = (
    combined_df[UNIFIED_COLUMNS]
    .apply(lambda col: col.apply(lambda x: x == "" or x == []))
    .mean()
    .sort_values(ascending=False)
    .round(3)
)

empty_report

origin_tags               0.834
origins                   0.834
manufacturing_places      0.802
generic_name              0.760
packaging                 0.551
packaging_tags            0.551
stores                    0.499
labels                    0.460
label_tags                0.460
ingredient_tags           0.399
ingredients_text          0.399
image_url                 0.168
category_tags             0.113
categories                0.113
ecoscore_grade            0.094
brand                     0.078
product_name              0.011
search_query              0.006
barcode                   0.000
source                    0.000
retrieval_method          0.000
countries                 0.000
country_tags              0.000
ecoscore_score            0.000
created_datetime          0.000
last_modified_datetime    0.000
api_status                0.000
extraction_timestamp      0.000
dtype: float64

In [97]:
# Top labels by source

def top_tags(df, source, column, n=20):
    values = []

    subset = df[df["source"] == source]

    for tags in subset[column]:
        values.extend(normalize_tags(tags))

    return pd.DataFrame(
        Counter(values).most_common(n),
        columns=["tag", "count"]
    )

In [98]:
top_tags(combined_df, "food", "label_tags", 20)

,tag,count
0,en:green-dot,75
1,en:nutriscore,70
2,en:vegetarian,66
3,en:rainforest-alliance,55
4,en:vegan,54
5,fr:triman,32
6,en:organic,32
7,en:eu-organic,32
8,en:no-gluten,31
9,en:made-in-germany,30


In [99]:
top_tags(combined_df, "beauty", "label_tags", 20)

,tag,count
0,en:vegetarian,21
1,en:vegan,21
2,en:the-vegan-society,7
3,en:made-in-germany,5
4,en:without-microplastics,4
5,en:without-silicon,2
6,en:green-point,2
7,de:Recyceltes Plastik,1
8,en:no-colorings,1
9,en:pefc,1


In [100]:
top_tags(combined_df, "products", "label_tags", 20)

,tag,count
0,en:made-in-germany,3
1,en:green-dot,2
2,en:repairability-index-8-2-france,1
3,en:fsc,1
4,en:fsc-mix,1
5,fr:triman,1
6,de:FSC C002400,1
7,de:FSC C118989,1
8,de:Ecolabel EE/006/00001,1
9,de:Ohne Mikroplastik,1


## 15. Create Scoring-Ready Sample ##

In [101]:
# Minimum useful fields for scoring

scoring_ready_df = combined_df[
    combined_df["has_product_name"]
    & combined_df["has_brand"]
    & combined_df["has_categories"]
].copy()

In [102]:
scoring_ready_df.shape

(442, 38)

In [103]:
scoring_ready_df["source"].value_counts()

source
food        285
beauty       99
products     58
Name: count, dtype: int64

## 16. Save Outputs ##

In [105]:
combined_df.to_csv("combined_openfacts_germany_etl_sample.csv", index=False)
quality_report.to_csv("data_quality_report_by_source.csv")
empty_report.to_csv("empty_values_report.csv")
scoring_ready_df.to_csv("scoring_ready_germany_sample.csv", index=False)

## 17. Final Markdown Conclusion ##

Day 2 Findings

We created reusable ETL functions for all three Open*Facts APIs.

Barcode extraction functions:
- `extract_food(barcode)`
- `extract_beauty(barcode)`
- `extract_product(barcode)`

Product search functions:
- `search_food(query)`
- `search_beauty(query)`
- `search_product(query)`

All API responses are transformed into one unified schema, regardless of source or retrieval method.

We filtered the combined dataset to products listed as available in Germany using `countries_tags`.

The final combined dataset includes standardized product names, brands, categories, labels, ingredients, packaging, countries, origins, manufacturing places, Eco-Score fields, retrieval metadata, API status, and extraction timestamps.

The data quality report shows that Open Food Facts provides the richest structured data, especially for ingredients and Eco-Score. Open Beauty Facts provides useful brand, category, and label information, but limited Eco-Score coverage. Open Products Facts is more sparse and should be used carefully or enriched with additional data.

This ETL notebook intentionally does not apply ESG scoring rules. It only extracts, standardizes, filters, and reports data quality. Scoring logic will be implemented in a separate scoring notebook.